<a href="https://colab.research.google.com/github/6pb4wnww4g-beep/Jason/blob/main/Jason%20Market%20Master%202.3.2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""Jason Market Master v2.3.2.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1QKn2hm2cDqThDQBH3xByQh5XcGfwT8ra
"""

# -*- coding: utf-8 -*-
"""Jason Market Master.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1QKn2hm2cDqThDQBH3xByQh5XcGfwT8ra
"""

# --- JASON MARKET MASTER v2.3.2 ---
# Formål:
# Samler hele market-kjeden i ett script.
#
# Kjør dette før Jason-hovedscriptet.
#
# Gjør alt dette:
# 1. Henter EPL-odds fra The Odds API
# 2. Skriver rå/ryddet OddsAPI-data
# 3. Beregner Market_xG_All_Fixtures_v1
# 4. Beregner Market_Team_Strength_v2_1
# 5. Beregner Market_Match_Strength_v2_1
#
# Skriver til Google Sheets:
# - OddsAPI_Raw_Matches
# - OddsAPI_Market_Check
# - OddsAPI_Bookmaker_Detail
# - Market_xG_All_Fixtures_v1
# - Market_xG_Parse_Check_v1
# - Market_Decimal_Check_v2_1
# - Market_Team_Strength_v2_1
# - Market_Match_Strength_v2_1
#
# Viktig:
# - Sett API_KEY før kjøring.
# - Scriptet henter alle EPL-kamper The Odds API returnerer/priser nå.
# - Ingen hardkodet GW1-5-liste.
# - Jason-hovedscriptet leser Market_Match_Strength_v2_1.

import time
import math
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from google.colab import auth
import gspread
from google.auth import default


# -----------------------
# KONFIGURASJON
# -----------------------

API_KEY = "1febfae459fe024b33c82de75598bd9f"
SPREADSHEET_NAME = "Jason Development"

SPORT_KEY = "soccer_epl"
REGIONS = "uk"
MARKETS = "h2h,totals"
ODDS_FORMAT = "decimal"
DATE_FORMAT = "iso"
BASE_URL = "https://api.the-odds-api.com/v4"

DEFAULT_TOTAL_GOALS = 2.65

MIN_SCALE = 10
MAX_SCALE = 95

PLANNER_GW_COUNT = 10
WIDE_SHEET_NAME = "Market_Projection_Wide_v1_1"
FPL_BOOTSTRAP_URL = "https://fantasy.premierleague.com/api/bootstrap-static/"
FPL_FIXTURES_URL = "https://fantasy.premierleague.com/api/fixtures/"


# -----------------------
# Google Sheets
# -----------------------

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
sh = gc.open(SPREADSHEET_NAME)


# -----------------------
# Generelle helpers
# -----------------------

def update_worksheet(sheet_name, dataframe, min_rows=200, min_cols=30, sleep_seconds=2):
    print(f"Oppdaterer {sheet_name}...")
    df_clean = dataframe.replace([np.inf, -np.inf], np.nan).copy()

    # gspread/simplejson kan ikke serialisere pandas Timestamp direkte.
    # Konverter alle datetime-kolonner og eventuelle Timestamp-objekter til ISO-tekst.
    for col in df_clean.columns:
        if pd.api.types.is_datetime64_any_dtype(df_clean[col]):
            df_clean[col] = df_clean[col].apply(
                lambda v: "" if pd.isna(v) else v.isoformat()
            )
        else:
            df_clean[col] = df_clean[col].apply(
                lambda v: v.isoformat()
                if isinstance(v, (pd.Timestamp, datetime))
                else v
            )

    df_clean = df_clean.fillna("")

    try:
        ws = sh.worksheet(sheet_name)
        ws.clear()
    except Exception:
        ws = sh.add_worksheet(title=sheet_name, rows=str(min_rows), cols=str(min_cols))

    ws.resize(
        rows=max(min_rows, len(df_clean) + 20),
        cols=max(min_cols, len(df_clean.columns) + 5)
    )

    ws.update([df_clean.columns.tolist()] + df_clean.values.tolist())
    time.sleep(sleep_seconds)


def parse_float(v, default=np.nan):
    if v is None:
        return default

    s = str(v).strip()
    if s == "":
        return default

    s = s.replace(" ", "")

    # Norsk desimalkomma
    if "," in s and "." not in s:
        s = s.replace(",", ".")

    # Hvis både komma og punktum finnes: anta komma som tusenskiller
    elif "," in s and "." in s:
        s = s.replace(",", "")

    try:
        return float(s)
    except Exception:
        return default


def get_remaining_headers(response):
    return {
        "x-requests-remaining": response.headers.get("x-requests-remaining", ""),
        "x-requests-used": response.headers.get("x-requests-used", ""),
        "x-requests-last": response.headers.get("x-requests-last", "")
    }


def norm_team(name):
    s = str(name).strip()
    aliases = {
        "Man City": "Manchester City",
        "Man United": "Manchester United",
        "Man Utd": "Manchester United",
        "Spurs": "Tottenham Hotspur",
        "Tottenham": "Tottenham Hotspur",
        "Brighton": "Brighton and Hove Albion",
        "Brighton & Hove Albion": "Brighton and Hove Albion",
        "AFC Bournemouth": "Bournemouth",
        "Bournemouth": "Bournemouth",
        "Newcastle": "Newcastle United",
        "Nott'm Forest": "Nottingham Forest",
        "Nottm Forest": "Nottingham Forest",
        "West Ham": "West Ham United",
        "Wolverhampton Wanderers": "Wolves",
        "Wolverhampton": "Wolves",
        "Ipswich": "Ipswich Town",
        "Leeds": "Leeds United",
        "Coventry": "Coventry City",
        "Hull": "Hull City",
    }
    return aliases.get(s, s)


# -----------------------
# DEL 1: Hent OddsAPI-data
# -----------------------

def fetch_oddsapi_events():
    if not API_KEY.strip() or API_KEY == "LIM_INN_DIN_API_KEY_HER":
        raise ValueError('API_KEY mangler. Lim inn nøkkelen din i API_KEY = "..."')

    url = f"{BASE_URL}/sports/{SPORT_KEY}/odds"
    params = {
        "apiKey": API_KEY,
        "regions": REGIONS,
        "markets": MARKETS,
        "oddsFormat": ODDS_FORMAT,
        "dateFormat": DATE_FORMAT,
    }

    print("Henter odds fra The Odds API...")
    response = requests.get(url, params=params, timeout=30)

    print("Status code:", response.status_code)
    print("Quota:", get_remaining_headers(response))

    if response.status_code != 200:
        raise RuntimeError(f"OddsAPI-feil {response.status_code}: {response.text[:1000]}")

    events = response.json()
    print(f"Antall events returnert fra OddsAPI: {len(events)}")

    return events


def flatten_oddsapi_events(events):
    raw_rows = []
    market_check_rows = []
    detail_rows = []

    for event in events:
        match_id = event.get("id", "")
        sport_key = event.get("sport_key", "")
        sport_title = event.get("sport_title", "")
        commence_time = event.get("commence_time", "")
        home_team = norm_team(event.get("home_team", ""))
        away_team = norm_team(event.get("away_team", ""))
        bookmakers = event.get("bookmakers", [])

        raw_rows.append({
            "match_id": match_id,
            "sport_key": sport_key,
            "sport_title": sport_title,
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers_count": len(bookmakers),
            "bookmakers": ", ".join([b.get("title", "") for b in bookmakers])
        })

        h2h_home_prices, h2h_draw_prices, h2h_away_prices = [], [], []
        totals_over_prices, totals_under_prices, totals_points = [], [], []
        markets_found = set()

        for bookmaker in bookmakers:
            bm_key = bookmaker.get("key", "")
            bm_title = bookmaker.get("title", "")
            bm_last_update = bookmaker.get("last_update", "")

            for market in bookmaker.get("markets", []):
                market_key = market.get("key", "")
                markets_found.add(market_key)
                market_last_update = market.get("last_update", "")

                for outcome in market.get("outcomes", []):
                    outcome_name = outcome.get("name", "")
                    price = outcome.get("price", "")
                    point = outcome.get("point", "")
                    description = outcome.get("description", "")

                    detail_rows.append({
                        "match_id": match_id,
                        "sport_key": sport_key,
                        "sport_title": sport_title,
                        "commence_time": commence_time,
                        "home_team": home_team,
                        "away_team": away_team,
                        "bookmaker_key": bm_key,
                        "bookmaker": bm_title,
                        "bookmaker_last_update": bm_last_update,
                        "market": market_key,
                        "market_last_update": market_last_update,
                        "outcome_name": outcome_name,
                        "price": price,
                        "point": point,
                        "description": description,
                    })

                    p = parse_float(price)
                    if pd.isna(p):
                        continue

                    if market_key == "h2h":
                        if outcome_name == home_team:
                            h2h_home_prices.append(p)
                        elif outcome_name == away_team:
                            h2h_away_prices.append(p)
                        elif str(outcome_name).lower() == "draw":
                            h2h_draw_prices.append(p)

                    elif market_key == "totals":
                        if point not in [None, ""]:
                            totals_points.append(point)
                        if str(outcome_name).lower() == "over":
                            totals_over_prices.append(p)
                        elif str(outcome_name).lower() == "under":
                            totals_under_prices.append(p)

        def avg(xs):
            return round(sum(xs) / len(xs), 3) if xs else ""

        def unique_points(xs):
            clean = sorted(list(set([x for x in xs if x not in [None, ""]])))
            return ", ".join([str(x) for x in clean])

        market_check_rows.append({
            "match_id": match_id,
            "commence_time": commence_time,
            "home_team": home_team,
            "away_team": away_team,
            "bookmakers_count": len(bookmakers),
            "markets_found": ", ".join(sorted(markets_found)),
            "avg_home_odds": avg(h2h_home_prices),
            "avg_draw_odds": avg(h2h_draw_prices),
            "avg_away_odds": avg(h2h_away_prices),
            "avg_over_odds": avg(totals_over_prices),
            "avg_under_odds": avg(totals_under_prices),
            "totals_lines_found": unique_points(totals_points),
            "h2h_books": len(h2h_home_prices),
            "totals_books": len(totals_over_prices),
        })

    raw_df = pd.DataFrame(raw_rows)
    market_check_df = pd.DataFrame(market_check_rows)
    detail_df = pd.DataFrame(detail_rows)

    if not raw_df.empty:
        raw_df = raw_df.sort_values(["commence_time", "home_team"]).reset_index(drop=True)

    if not market_check_df.empty:
        market_check_df = market_check_df.sort_values(["commence_time", "home_team"]).reset_index(drop=True)

    if not detail_df.empty:
        detail_df = detail_df.sort_values(
            ["commence_time", "home_team", "bookmaker", "market", "outcome_name"]
        ).reset_index(drop=True)

    return raw_df, market_check_df, detail_df


# -----------------------
# DEL 2: Odds -> kamp-xG
# -----------------------

def avg_no_vig_probs(prices):
    implied = {}

    for outcome, odds_list in prices.items():
        clean = [parse_float(x) for x in odds_list]
        clean = [x for x in clean if not pd.isna(x) and x > 1.0]

        if clean:
            avg_odds = sum(clean) / len(clean)
            implied[outcome] = 1.0 / avg_odds

    total = sum(implied.values())

    if total <= 0:
        return {}

    return {k: v / total for k, v in implied.items()}


def poisson_pmf(lam, k):
    return math.exp(-lam) * (lam ** k) / math.factorial(k)


def poisson_match_probs(home_xg, away_xg, max_goals=10):
    home_win = 0.0
    draw = 0.0
    away_win = 0.0
    total_under_25 = 0.0

    home_zero = poisson_pmf(home_xg, 0)
    away_zero = poisson_pmf(away_xg, 0)

    home_cs = away_zero
    away_cs = home_zero

    for h in range(max_goals + 1):
        ph = poisson_pmf(home_xg, h)

        for a in range(max_goals + 1):
            pa = poisson_pmf(away_xg, a)
            p = ph * pa

            if h > a:
                home_win += p
            elif h == a:
                draw += p
            else:
                away_win += p

            if h + a <= 2:
                total_under_25 += p

    return {
        "home_win": home_win,
        "draw": draw,
        "away_win": away_win,
        "home_cs": home_cs,
        "away_cs": away_cs,
        "under_25": total_under_25,
        "over_25": 1.0 - total_under_25,
    }


def expected_total_from_over25(over_prob):
    if over_prob is None or pd.isna(over_prob):
        return DEFAULT_TOTAL_GOALS

    target = max(0.05, min(0.95, float(over_prob)))

    best_lam = DEFAULT_TOTAL_GOALS
    best_err = 999

    for lam in np.arange(0.60, 5.51, 0.01):
        p_under_25 = sum(poisson_pmf(lam, k) for k in range(0, 3))
        p_over_25 = 1.0 - p_under_25
        err = abs(p_over_25 - target)

        if err < best_err:
            best_err = err
            best_lam = lam

    return float(best_lam)


def fit_xg_from_market(home_win_p, draw_p, away_win_p, total_goals):
    total_goals = max(0.6, min(5.5, float(total_goals)))

    best = {
        "home_xg": total_goals / 2,
        "away_xg": total_goals / 2,
        "err": 999,
        "probs": None,
    }

    for home_share in np.arange(0.15, 0.86, 0.0025):
        hxg = total_goals * home_share
        axg = total_goals - hxg

        probs = poisson_match_probs(hxg, axg)

        err = (
            abs(probs["home_win"] - home_win_p) +
            abs(probs["draw"] - draw_p) +
            abs(probs["away_win"] - away_win_p)
        )

        if err < best["err"]:
            best = {
                "home_xg": hxg,
                "away_xg": axg,
                "err": err,
                "probs": probs,
            }

    return best


def build_market_xg(detail_df):
    required = [
        "match_id",
        "commence_time",
        "home_team",
        "away_team",
        "bookmaker",
        "market",
        "outcome_name",
        "price",
    ]

    missing = [c for c in required if c not in detail_df.columns]
    if missing:
        raise ValueError(f"Input mangler kolonner: {missing}")

    if "point" not in detail_df.columns:
        detail_df["point"] = ""

    rows = []
    check_rows = []

    grouped = detail_df.groupby(["match_id", "commence_time", "home_team", "away_team"], dropna=False)

    for (match_id, kickoff, home, away), g in grouped:
        home = str(home).strip()
        away = str(away).strip()

        h2h = g[g["market"].astype(str).str.lower().eq("h2h")].copy()
        totals = g[g["market"].astype(str).str.lower().eq("totals")].copy()

        h2h_prices = {
            "home": [],
            "draw": [],
            "away": [],
        }

        for _, r in h2h.iterrows():
            outcome = str(r.get("outcome_name", "")).strip()
            price = r.get("price", "")

            if outcome == home:
                h2h_prices["home"].append(price)
            elif outcome == away:
                h2h_prices["away"].append(price)
            elif outcome.lower() == "draw":
                h2h_prices["draw"].append(price)

        h2h_probs = avg_no_vig_probs(h2h_prices)

        home_p = h2h_probs.get("home", np.nan)
        draw_p = h2h_probs.get("draw", np.nan)
        away_p = h2h_probs.get("away", np.nan)

        over_prices = []
        under_prices = []
        selected_point = np.nan

        if not totals.empty:
            totals["point_num"] = totals["point"].apply(parse_float)
            points = sorted([p for p in totals["point_num"].dropna().unique().tolist()])

            if points:
                selected_point = min(points, key=lambda x: abs(x - 2.5))
                tsel = totals[totals["point_num"].eq(selected_point)]

                for _, r in tsel.iterrows():
                    outcome = str(r.get("outcome_name", "")).strip().lower()
                    price = r.get("price", "")

                    if outcome == "over":
                        over_prices.append(price)
                    elif outcome == "under":
                        under_prices.append(price)

        total_probs = avg_no_vig_probs({
            "over": over_prices,
            "under": under_prices,
        })

        over_p = total_probs.get("over", np.nan)
        total_goals = expected_total_from_over25(over_p)

        if any(pd.isna(x) for x in [home_p, draw_p, away_p]):
            check_rows.append({
                "match_id": match_id,
                "home_team": home,
                "away_team": away,
                "status": "MISSING_H2H",
                "h2h_rows": len(h2h),
                "totals_rows": len(totals),
            })
            continue

        fit = fit_xg_from_market(home_p, draw_p, away_p, total_goals)
        hxg = fit["home_xg"]
        axg = fit["away_xg"]
        probs = fit["probs"] or poisson_match_probs(hxg, axg)

        rows.append({
            "Kickoff": kickoff,
            "Team": home,
            "Opponent": away,
            "H/A": "H",
            "Team xG": round(hxg, 4),
            "Opp xG": round(axg, 4),
            "CS%": round(probs["home_cs"] * 100, 4),
            "Win%": round(probs["home_win"] * 100, 4),
            "Draw%": round(probs["draw"] * 100, 4),
            "Loss%": round(probs["away_win"] * 100, 4),
            "Total xG": round(hxg + axg, 4),
            "Fit Error": round(fit["err"], 4),
            "Match ID": match_id,
        })

        rows.append({
            "Kickoff": kickoff,
            "Team": away,
            "Opponent": home,
            "H/A": "A",
            "Team xG": round(axg, 4),
            "Opp xG": round(hxg, 4),
            "CS%": round(probs["away_cs"] * 100, 4),
            "Win%": round(probs["away_win"] * 100, 4),
            "Draw%": round(probs["draw"] * 100, 4),
            "Loss%": round(probs["home_win"] * 100, 4),
            "Total xG": round(hxg + axg, 4),
            "Fit Error": round(fit["err"], 4),
            "Match ID": match_id,
        })

        check_rows.append({
            "match_id": match_id,
            "kickoff": kickoff,
            "home_team": home,
            "away_team": away,
            "status": "OK",
            "h2h_home_p": round(home_p, 4),
            "h2h_draw_p": round(draw_p, 4),
            "h2h_away_p": round(away_p, 4),
            "selected_total_line": selected_point,
            "over_p": round(over_p, 4) if not pd.isna(over_p) else "",
            "total_goals_est": round(total_goals, 4),
            "home_xg": round(hxg, 4),
            "away_xg": round(axg, 4),
            "fit_error": round(fit["err"], 4),
            "h2h_rows": len(h2h),
            "totals_over_rows": len(over_prices),
            "totals_under_rows": len(under_prices),
        })

    out_df = pd.DataFrame(rows)
    check_df = pd.DataFrame(check_rows)

    if not out_df.empty:
        out_df = out_df.sort_values(["Kickoff", "Team"]).reset_index(drop=True)

    if not check_df.empty:
        check_df = check_df.sort_values(["kickoff", "home_team"], na_position="last").reset_index(drop=True)

    return out_df, check_df


# -----------------------
# DEL 3: kamp-xG -> Team Strength / Match Strength
# -----------------------

def scale_10_95(series):
    s = pd.to_numeric(series, errors="coerce")
    mn = s.min()
    mx = s.max()

    if pd.isna(mn) or pd.isna(mx) or mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)

    return (MIN_SCALE + (s - mn) / (mx - mn) * (MAX_SCALE - MIN_SCALE)).clip(MIN_SCALE, MAX_SCALE)


def build_team_and_match_strength(market_xg_df):
    df = market_xg_df.copy()

    required = [
        "Kickoff", "Team", "Opponent", "H/A",
        "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Mangler kolonner i Market_xG_All_Fixtures_v1: {missing}")

    check_df = df[[
        "Kickoff", "Team", "Opponent", "H/A",
        "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%"
    ]].copy()

    check_df = check_df.rename(columns={
        "Team xG": "Raw Team xG",
        "Opp xG": "Raw Opp xG",
        "CS%": "Raw CS%",
        "Win%": "Raw Win%",
        "Draw%": "Raw Draw%",
        "Loss%": "Raw Loss%"
    })

    for col in ["Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG", "Fit Error"]:
        if col in df.columns:
            df[col] = df[col].apply(parse_float)

    df = df.dropna(subset=["Team xG", "Opp xG"]).copy()

    if df.empty:
        raise ValueError("Ingen gyldige xG-rader funnet i Market_xG_All_Fixtures_v1")

    check_df = check_df.loc[df.index].copy()
    check_df["Parsed Team xG"] = df["Team xG"].values
    check_df["Parsed Opp xG"] = df["Opp xG"].values
    check_df["Check Note"] = ""

    league_avg_xg = df["Team xG"].mean()

    team = (
        df.groupby("Team", as_index=False)
        .agg({
            "Team xG": ["mean", "sum", "count"],
            "Opp xG": ["mean", "sum"],
            "CS%": "mean",
            "Win%": "mean",
            "Draw%": "mean",
            "Loss%": "mean"
        })
    )

    team.columns = [
        "Team",
        "Avg Team xG",
        "Total Team xG",
        "Matches",
        "Avg Opp xG",
        "Total Opp xG",
        "Avg CS%",
        "Avg Win%",
        "Avg Draw%",
        "Avg Loss%"
    ]

    team["League Avg xG"] = league_avg_xg
    team["Team Att Ratio"] = team["Avg Team xG"] / league_avg_xg
    team["Team Def Ratio"] = league_avg_xg / team["Avg Opp xG"]

    team["Team Att"] = scale_10_95(team["Team Att Ratio"])
    team["Team Def"] = scale_10_95(team["Team Def Ratio"])

    team["Interpretation"] = team.apply(
        lambda r: f"Att {r['Team Att Ratio']:.2f}x league avg, Def {r['Team Def Ratio']:.2f}x league avg",
        axis=1
    )

    team_out = team[[
        "Team",
        "Matches",
        "Avg Team xG",
        "Avg Opp xG",
        "League Avg xG",
        "Team Att Ratio",
        "Team Def Ratio",
        "Team Att",
        "Team Def",
        "Avg CS%",
        "Avg Win%",
        "Avg Draw%",
        "Avg Loss%",
        "Total Team xG",
        "Total Opp xG",
        "Interpretation"
    ]].copy()

    team_att_map = dict(zip(team["Team"], team["Team Att"]))
    team_def_map = dict(zip(team["Team"], team["Team Def"]))
    team_att_ratio_map = dict(zip(team["Team"], team["Team Att Ratio"]))
    team_def_ratio_map = dict(zip(team["Team"], team["Team Def Ratio"]))

    match = df.copy()

    match["Team Att"] = match["Team"].map(team_att_map)
    match["Team Def"] = match["Team"].map(team_def_map)
    match["Opp Team Att"] = match["Opponent"].map(team_att_map)
    match["Opp Team Def"] = match["Opponent"].map(team_def_map)

    match["Team Att Ratio"] = match["Team"].map(team_att_ratio_map)
    match["Team Def Ratio"] = match["Team"].map(team_def_ratio_map)
    match["Opp Team Att Ratio"] = match["Opponent"].map(team_att_ratio_map)
    match["Opp Team Def Ratio"] = match["Opponent"].map(team_def_ratio_map)

    match["Match Att"] = match["Team Att"] - match["Opp Team Def"]
    match["Match Def"] = match["Team Def"] - match["Opp Team Att"]

    match["Match Att Ratio Diff"] = match["Team Att Ratio"] - match["Opp Team Def Ratio"]
    match["Match Def Ratio Diff"] = match["Team Def Ratio"] - match["Opp Team Att Ratio"]

    match_out_cols = [
        "Kickoff",
        "Team",
        "Opponent",
        "H/A",
        "Team Att",
        "Opp Team Def",
        "Match Att",
        "Team xG",
        "Team Def",
        "Opp Team Att",
        "Match Def",
        "Opp xG",
        "CS%",
        "Win%",
        "Draw%",
        "Loss%",
        "Team Att Ratio",
        "Opp Team Def Ratio",
        "Match Att Ratio Diff",
        "Team Def Ratio",
        "Opp Team Att Ratio",
        "Match Def Ratio Diff"
    ]

    match_out = match[match_out_cols].copy()

    for out_df in [check_df, team_out, match_out]:
        for col in out_df.columns:
            if pd.api.types.is_numeric_dtype(out_df[col]):
                out_df[col] = out_df[col].round(4)

    team_out = team_out.sort_values("Team Att", ascending=False).reset_index(drop=True)
    match_out = match_out.sort_values(["Kickoff", "Team"]).reset_index(drop=True)

    return check_df, team_out, match_out, league_avg_xg



# -----------------------
# DEL 4: Team Strength -> 10 GW Projection Wide
# -----------------------

def fetch_fpl_context(market_xg_df=None):
    """Kobler OddsAPI sin neste kamp eksplisitt mot korrekt FPL-fixture.

    Market har bare data for neste kamp. Derfor bestemmes start-GW fra de
    kampene som faktisk matcher på hjemmelag, bortelag og nærmeste kickoff.
    Bootstrap-feltene is_next/finished brukes bare som fallback.
    """
    print("Henter FPL terminliste og kobler Market til korrekt kamp...")

    bootstrap_response = requests.get(FPL_BOOTSTRAP_URL, timeout=30)
    fixtures_response = requests.get(FPL_FIXTURES_URL, timeout=30)
    bootstrap_response.raise_for_status()
    fixtures_response.raise_for_status()

    bootstrap = bootstrap_response.json()
    fixtures = fixtures_response.json()

    teams_by_id = {}
    for team in bootstrap.get("teams", []):
        team_id = int(team["id"])
        teams_by_id[team_id] = {
            "name": norm_team(team.get("name", "")),
            "code": str(team.get("short_name", "")).upper(),
        }

    fpl_rows = []
    for fixture in fixtures:
        event = fixture.get("event")
        home_id = fixture.get("team_h")
        away_id = fixture.get("team_a")
        if event is None or home_id not in teams_by_id or away_id not in teams_by_id:
            continue
        fpl_rows.append({
            "FPL Fixture ID": int(fixture.get("id")),
            "Actual GW": int(event),
            "FPL Kickoff": pd.to_datetime(fixture.get("kickoff_time"), utc=True, errors="coerce"),
            "Home Team": teams_by_id[home_id]["name"],
            "Away Team": teams_by_id[away_id]["name"],
            "Home Code": teams_by_id[home_id]["code"],
            "Away Code": teams_by_id[away_id]["code"],
            "Finished": bool(fixture.get("finished", False)),
        })
    fpl_df = pd.DataFrame(fpl_rows)
    if fpl_df.empty:
        raise ValueError("FPL-terminlisten inneholder ingen brukbare kamper.")

    # Én rad per OddsAPI-kamp (hjemmeraden er nok til kampidentitet).
    market_matches = []
    if market_xg_df is not None and not market_xg_df.empty:
        home_rows = market_xg_df[
            market_xg_df["H/A"].astype(str).str.upper().eq("H")
        ].copy()
        for _, row in home_rows.iterrows():
            market_matches.append({
                "Market Match ID": str(row.get("Match ID", "")).strip(),
                "Market Kickoff": pd.to_datetime(row.get("Kickoff"), utc=True, errors="coerce"),
                "Home Team": norm_team(row.get("Team", "")),
                "Away Team": norm_team(row.get("Opponent", "")),
            })

    matched = []
    unmatched = []
    used_fixture_ids = set()
    max_time_diff_hours = 72.0

    for mm in market_matches:
        candidates = fpl_df[
            fpl_df["Home Team"].eq(mm["Home Team"])
            & fpl_df["Away Team"].eq(mm["Away Team"])
            & ~fpl_df["Finished"]
        ].copy()

        if candidates.empty:
            unmatched.append({**mm, "Reason": "ingen uspilt FPL-kamp med samme hjem/borte"})
            continue

        if pd.notna(mm["Market Kickoff"]):
            candidates["Time Difference Hours"] = (
                candidates["FPL Kickoff"] - mm["Market Kickoff"]
            ).abs().dt.total_seconds() / 3600.0
            candidates = candidates.sort_values(["Time Difference Hours", "Actual GW"])
        else:
            candidates["Time Difference Hours"] = np.nan
            candidates = candidates.sort_values(["Actual GW", "FPL Kickoff"])

        best = candidates.iloc[0]
        diff = best["Time Difference Hours"]
        if pd.notna(diff) and float(diff) > max_time_diff_hours:
            unmatched.append({**mm, "Reason": f"kickoff-avvik {float(diff):.1f} timer"})
            continue
        if int(best["FPL Fixture ID"]) in used_fixture_ids:
            unmatched.append({**mm, "Reason": "FPL-kampen var allerede koblet"})
            continue

        used_fixture_ids.add(int(best["FPL Fixture ID"]))
        matched.append({
            **mm,
            "FPL Fixture ID": int(best["FPL Fixture ID"]),
            "Actual GW": int(best["Actual GW"]),
            "FPL Kickoff": best["FPL Kickoff"],
            "Time Difference Hours": float(diff) if pd.notna(diff) else np.nan,
            "Status": "MATCHED",
        })

    match_df = pd.DataFrame(matched)
    unmatched_df = pd.DataFrame(unmatched)

    if not match_df.empty:
        start_gw = int(match_df["Actual GW"].min())
    else:
        # Fallback dersom OddsAPI midlertidig ikke returnerer brukbare kamper.
        events = bootstrap.get("events", [])
        next_events = [e for e in events if e.get("is_next")]
        if next_events:
            start_gw = int(next_events[0]["id"])
        else:
            future = fpl_df[(~fpl_df["Finished"]) & fpl_df["FPL Kickoff"].notna()].copy()
            if future.empty:
                raise ValueError("Ingen Market-kamp kunne kobles og ingen framtidig FPL-kamp ble funnet.")
            start_gw = int(future.sort_values("FPL Kickoff").iloc[0]["Actual GW"])

    schedule_df = fpl_df[
        (fpl_df["Actual GW"] >= start_gw)
        & (fpl_df["Actual GW"] < start_gw + PLANNER_GW_COUNT)
    ].copy()
    if schedule_df.empty:
        raise ValueError(f"Fant ingen FPL-kamper fra GW{start_gw}.")

    schedule_df["Market Match ID"] = ""
    schedule_df["Market Match Status"] = "PROJECTION"
    schedule_df["Market Kickoff"] = pd.NaT
    schedule_df["Time Difference Hours"] = np.nan

    if not match_df.empty:
        by_fixture = match_df.set_index("FPL Fixture ID")
        for idx, row in schedule_df.iterrows():
            fixture_id = int(row["FPL Fixture ID"])
            if fixture_id in by_fixture.index:
                m = by_fixture.loc[fixture_id]
                schedule_df.at[idx, "Market Match ID"] = str(m["Market Match ID"])
                schedule_df.at[idx, "Market Match Status"] = "MATCHED"
                schedule_df.at[idx, "Market Kickoff"] = m["Market Kickoff"]
                schedule_df.at[idx, "Time Difference Hours"] = m["Time Difference Hours"]

    print("\n=== MARKET → FPL FIXTURE-KONTROLL ===")
    print(f"Market-kamper: {len(market_matches)}")
    print(f"Matchet: {len(match_df)}")
    print(f"Umatchet: {len(unmatched_df)}")
    print(f"Faktisk start-GW: GW{start_gw}")
    if not match_df.empty:
        cols = ["Home Team", "Away Team", "Actual GW", "FPL Fixture ID", "Time Difference Hours"]
        # Sorter mens FPL Kickoff fortsatt finnes i DataFrame; vis deretter kontrollkolonnene.
        match_print = match_df.sort_values(["Actual GW", "FPL Kickoff"])[cols]
        print(match_print.to_string(index=False))
    if not unmatched_df.empty:
        print("\nADVARSEL – umatchet Market-kamper:")
        print(unmatched_df[["Home Team", "Away Team", "Reason"]].to_string(index=False))

    # Eget kontrollark gjør feil kobling synlig før Jason bruker resultatet.
    audit_rows = []
    for _, r in match_df.iterrows():
        audit_rows.append({
            "Status": "MATCHED",
            "Market Match ID": r["Market Match ID"],
            "Home Team": r["Home Team"],
            "Away Team": r["Away Team"],
            "Market Kickoff": r["Market Kickoff"],
            "FPL Fixture ID": r["FPL Fixture ID"],
            "Actual GW": r["Actual GW"],
            "FPL Kickoff": r["FPL Kickoff"],
            "Time Difference Hours": r["Time Difference Hours"],
            "Reason": "",
        })
    for _, r in unmatched_df.iterrows():
        audit_rows.append({
            "Status": "UNMATCHED",
            "Market Match ID": r.get("Market Match ID", ""),
            "Home Team": r.get("Home Team", ""),
            "Away Team": r.get("Away Team", ""),
            "Market Kickoff": r.get("Market Kickoff", ""),
            "FPL Fixture ID": "",
            "Actual GW": "",
            "FPL Kickoff": "",
            "Time Difference Hours": "",
            "Reason": r.get("Reason", ""),
        })
    audit_df = pd.DataFrame(audit_rows)
    if not audit_df.empty:
        update_worksheet("Market_Fixture_Match_Check_v2_3", audit_df, min_rows=100, min_cols=15)

    team_code_map = {v["name"]: v["code"] for v in teams_by_id.values()}
    team_names = sorted(v["name"] for v in teams_by_id.values())
    return schedule_df, team_code_map, team_names, start_gw


def build_market_lookup(market_xg_df):
    """Lookup per OddsAPI Match ID og lagperspektiv."""
    lookup = {}
    for _, row in market_xg_df.iterrows():
        match_id = str(row.get("Match ID", "")).strip()
        team = norm_team(row.get("Team", ""))
        ha = str(row.get("H/A", "")).strip().upper()
        if not match_id or not team or ha not in {"H", "A"}:
            continue
        lookup[(match_id, team, ha)] = row.to_dict()
    return lookup

def infer_venue_factors(market_xg_df, league_avg_xg):
    """Kalibrerer hjemme-/bortefaktor fra oddsene i førstkommende GW."""
    df = market_xg_df.copy()
    df["Team xG"] = pd.to_numeric(df["Team xG"], errors="coerce")
    home_mean = df.loc[df["H/A"].astype(str).str.upper().eq("H"), "Team xG"].mean()
    away_mean = df.loc[df["H/A"].astype(str).str.upper().eq("A"), "Team xG"].mean()

    if pd.isna(home_mean) or pd.isna(away_mean) or league_avg_xg <= 0:
        return 1.10, 0.90

    return float(home_mean / league_avg_xg), float(away_mean / league_avg_xg)


def project_fixture_from_strength(home, away, team_strength_map, league_avg_xg,
                                  home_factor, away_factor):
    """Projiserer én framtidig kamp fra lagstyrkene hentet fra førstkommende GW."""
    h = team_strength_map.get(home)
    a = team_strength_map.get(away)
    if h is None or a is None:
        return None

    h_att = max(0.05, float(h["Team Att Ratio"]))
    h_def = max(0.05, float(h["Team Def Ratio"]))
    a_att = max(0.05, float(a["Team Att Ratio"]))
    a_def = max(0.05, float(a["Team Def Ratio"]))

    home_xg = league_avg_xg * (h_att / a_def) * home_factor
    away_xg = league_avg_xg * (a_att / h_def) * away_factor

    home_xg = max(0.05, min(5.50, home_xg))
    away_xg = max(0.05, min(5.50, away_xg))
    probs = poisson_match_probs(home_xg, away_xg)

    return {
        "home": {
            "Team xG": home_xg,
            "Opp xG": away_xg,
            "CS%": probs["home_cs"] * 100,
            "Win%": probs["home_win"] * 100,
            "Draw%": probs["draw"] * 100,
            "Loss%": probs["away_win"] * 100,
            "Total xG": home_xg + away_xg,
        },
        "away": {
            "Team xG": away_xg,
            "Opp xG": home_xg,
            "CS%": probs["away_cs"] * 100,
            "Win%": probs["away_win"] * 100,
            "Draw%": probs["draw"] * 100,
            "Loss%": probs["home_win"] * 100,
            "Total xG": home_xg + away_xg,
        },
    }


def build_projection_wide_10gw(market_xg_df, team_strength_df, league_avg_xg,
                               schedule_df, team_code_map, team_names, start_gw):
    """
    GW1 er alltid førstkommende faktiske FPL-GW.
    GW1 bruker OddsAPI-verdiene direkte.
    GW2-GW10 beregnes fra GW1-lagstyrkene og offisiell FPL-terminliste.
    """
    market_lookup = build_market_lookup(market_xg_df)
    team_strength_map = {
        norm_team(r["Team"]): {
            "Team Att Ratio": parse_float(r["Team Att Ratio"]),
            "Team Def Ratio": parse_float(r["Team Def Ratio"]),
        }
        for _, r in team_strength_df.iterrows()
    }
    home_factor, away_factor = infer_venue_factors(market_xg_df, league_avg_xg)

    fixture_data = {}
    unmatched_gw1 = []

    for _, fx in schedule_df.iterrows():
        actual_gw = int(fx["Actual GW"])
        display_gw = actual_gw - start_gw + 1
        home = norm_team(fx["Home Team"])
        away = norm_team(fx["Away Team"])

        market_match_id = str(fx.get("Market Match ID", "")).strip()
        if market_match_id:
            home_market = market_lookup.get((market_match_id, home, "H"))
            away_market = market_lookup.get((market_match_id, away, "A"))
            if home_market is None or away_market is None:
                unmatched_gw1.append(f"{home} - {away} (fixture {fx.get('FPL Fixture ID', '')})")
                projected = project_fixture_from_strength(
                    home, away, team_strength_map, league_avg_xg, home_factor, away_factor
                )
                if projected is None:
                    continue
                home_values = projected["home"]
                away_values = projected["away"]
            else:
                home_values = {k: parse_float(home_market.get(k)) for k in [
                    "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG"
                ]}
                away_values = {k: parse_float(away_market.get(k)) for k in [
                    "Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG"
                ]}
        else:
            projected = project_fixture_from_strength(
                home, away, team_strength_map, league_avg_xg, home_factor, away_factor
            )
            if projected is None:
                continue
            home_values = projected["home"]
            away_values = projected["away"]

        fixture_data.setdefault((home, display_gw), []).append({
            "Opp": f"{fx['Away Code']} (H)", "H/A": "H", **home_values
        })
        fixture_data.setdefault((away, display_gw), []).append({
            "Opp": f"{fx['Home Code']} (A)", "H/A": "A", **away_values
        })

    if unmatched_gw1:
        print("ADVARSEL: Følgende koblede Market-kamper manglet direkte radoppslag og ble projisert:")
        for match in unmatched_gw1:
            print(" -", match)

    rows = []
    numeric_keys = ["Team xG", "Opp xG", "CS%", "Win%", "Draw%", "Loss%", "Total xG"]

    for team in team_names:
        row = {"Team Code": team_code_map.get(team, ""), "Team": team}
        for gw in range(1, PLANNER_GW_COUNT + 1):
            items = fixture_data.get((team, gw), [])
            prefix = f"GW{gw}"
            row[f"{prefix} Opp"] = " + ".join(i["Opp"] for i in items)
            row[f"{prefix} H/A"] = "+".join(i["H/A"] for i in items)
            for key in numeric_keys:
                vals = [parse_float(i.get(key)) for i in items]
                vals = [v for v in vals if not pd.isna(v)]
                row[f"{prefix} {key if key != 'Team xG' else 'xG'}"] = (
                    round(sum(vals), 8) if vals else ""
                )
        rows.append(row)

    columns = ["Team Code", "Team"]
    for gw in range(1, PLANNER_GW_COUNT + 1):
        columns += [
            f"GW{gw} Opp", f"GW{gw} H/A", f"GW{gw} xG", f"GW{gw} Opp xG",
            f"GW{gw} CS%", f"GW{gw} Win%", f"GW{gw} Draw%", f"GW{gw} Loss%",
            f"GW{gw} Total xG",
        ]

    return pd.DataFrame(rows, columns=columns), home_factor, away_factor

# -----------------------
# RUN
# -----------------------


events = fetch_oddsapi_events()

raw_df, market_check_df, detail_df = flatten_oddsapi_events(events)

update_worksheet("OddsAPI_Raw_Matches", raw_df, min_rows=1000, min_cols=80)
update_worksheet("OddsAPI_Market_Check", market_check_df, min_rows=1000, min_cols=80)
update_worksheet("OddsAPI_Bookmaker_Detail", detail_df, min_rows=2000, min_cols=80)

market_xg_df, parse_check_df = build_market_xg(detail_df)

update_worksheet("Market_xG_All_Fixtures_v1", market_xg_df, min_rows=2000, min_cols=80)
update_worksheet("Market_xG_Parse_Check_v1", parse_check_df, min_rows=1000, min_cols=80)

decimal_check_df, team_strength_df, match_strength_df, league_avg_xg = build_team_and_match_strength(market_xg_df)

update_worksheet("Market_Decimal_Check_v2_1", decimal_check_df, min_rows=1000, min_cols=80)
update_worksheet("Market_Team_Strength_v2_1", team_strength_df, min_rows=200, min_cols=80)
update_worksheet("Market_Match_Strength_v2_1", match_strength_df, min_rows=2000, min_cols=80)

schedule_df, team_code_map, team_names, start_gw = fetch_fpl_context(market_xg_df)
projection_wide_df, home_factor, away_factor = build_projection_wide_10gw(
    market_xg_df=market_xg_df,
    team_strength_df=team_strength_df,
    league_avg_xg=league_avg_xg,
    schedule_df=schedule_df,
    team_code_map=team_code_map,
    team_names=team_names,
    start_gw=start_gw,
)
update_worksheet(WIDE_SHEET_NAME, projection_wide_df, min_rows=200, min_cols=100)

print("")
print("FERDIG: Jason Market Master v2.1 (10 GW)")
print(f"Antall kamper fra OddsAPI: {len(raw_df)}")
print(f"Market_xG-rader: {len(market_xg_df)}")
print(f"Market_Match_Strength-rader: {len(match_strength_df)}")
print(f"League Avg xG i grunnlaget: {league_avg_xg:.4f}")
print(f"Market-koblet faktisk GW: GW{start_gw}")
print(f"Hjemmefaktor fra Odds-GW: {home_factor:.4f}")
print(f"Bortefaktor fra Odds-GW: {away_factor:.4f}")
print(f"{WIDE_SHEET_NAME}: faktisk GW{start_gw}-GW{start_gw + 9} skrevet som GW1-GW10")
print("")
print("GW1 = direkte OddsAPI. GW2-GW10 = projisert fra GW1-lagstyrkene.")
print("Neste steg: Kontroller Market_Projection_Wide_v1_1 før Jason-hovedscriptet kjøres.")

Henter odds fra The Odds API...
Status code: 200
Quota: {'x-requests-remaining': '482', 'x-requests-used': '18', 'x-requests-last': '2'}
Antall events returnert fra OddsAPI: 10
Oppdaterer OddsAPI_Raw_Matches...
Oppdaterer OddsAPI_Market_Check...
Oppdaterer OddsAPI_Bookmaker_Detail...
Oppdaterer Market_xG_All_Fixtures_v1...
Oppdaterer Market_xG_Parse_Check_v1...
Oppdaterer Market_Decimal_Check_v2_1...
Oppdaterer Market_Team_Strength_v2_1...
Oppdaterer Market_Match_Strength_v2_1...
Henter FPL terminliste og kobler Market til korrekt kamp...


/tmp/ipykernel_1448/1545512263.py:952: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '2026-08-21 19:00:00+00:00' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  schedule_df.at[idx, "Market Kickoff"] = m["Market Kickoff"]



=== MARKET → FPL FIXTURE-KONTROLL ===
Market-kamper: 10
Matchet: 10
Umatchet: 0
Faktisk start-GW: GW1
               Home Team         Away Team  Actual GW  FPL Fixture ID  Time Difference Hours
                 Arsenal     Coventry City          1               1                    0.0
               Hull City Manchester United          1               4                    0.0
                 Everton    Crystal Palace          1               3                    0.0
            Ipswich Town        Sunderland          1               5                    0.0
       Nottingham Forest      Leeds United          1               6                    0.0
               Brentford Tottenham Hotspur          1               2                    0.0
Brighton and Hove Albion       Aston Villa          1               7                    0.0
         Manchester City       Bournemouth          1               8                    0.0
        Newcastle United         Liverpool          1       